In [ ]:
import torch
import torch.nn as nn 

class MHA(nn.Module):

    def __init__(self, embed_dim, heads):
        super().__init__()

        assert embed_dim % heads == 0

        self.embed_dim = embed_dim
        self.heads = heads
        self.head_dim = embed_dim // heads

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, causal):

        B, T, E = x.shape
        # Each is (B, T, embed_dim)
        q, k, v = self.qkv(x).chunk(3, dim = -1)

        # (B, T, heads, head_dim)
        q = q.reshape(B, T, self.heads, -1)
        k = k.reshape(B, T, self.heads, -1)
        v = v.reshape(B, T, self.heads, -1)

        # (B, heads, T, head_dim)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) / (self.head_dim ** 0.5)

        if causal:

            mask = torch.tril(torch.ones((T, T), dtype = torch.bool))
            scores = scores.masked_fill(~mask, float("-inf"))

        p = torch.softmax(scores, dim = -1)
        # (B, heads, T, head_dim)
        o = p @ v 
        output = self.proj(o.transpose(1, 2).reshape(B, T, -1))
        return output
        
